## LDA ON OLIVETTI DATASET (FROM SCRATCH)

In [15]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

In [16]:
# Load dataset
faces = fetch_olivetti_faces()
X = faces.data
y = faces.target

In [17]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [18]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [19]:
# Random forest on raw data
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_scaled, y_train)
print("Accuracy on raw data:",
      metrics.accuracy_score(y_test, clf.predict(X_test_scaled)))

Accuracy on raw data: 0.9375


In [20]:
# Logistic regression on raw data
lr_raw = LogisticRegression(max_iter=2000)
lr_raw.fit(X_train_scaled, y_train)
print("Accuracy using Logistic Regression (Raw):",
      metrics.accuracy_score(y_test, lr_raw.predict(X_test_scaled)))

Accuracy using Logistic Regression (Raw): 0.9625


**Manual LDA**

In [21]:
# Compute means
classes = np.unique(y_train)
n_features = X_train_scaled.shape[1]
mean_overall = np.mean(X_train_scaled, axis=0)
S_W = np.zeros((n_features, n_features))
S_B = np.zeros((n_features, n_features))

In [22]:
# Compute scatter matrices
for c in classes:
    X_c = X_train_scaled[y_train == c]
    mean_c = np.mean(X_c, axis=0)
    # Within-class scatter
    S_W += (X_c - mean_c).T @ (X_c - mean_c)
    # Between-class scatter
    n_c = X_c.shape[0]
    mean_diff = (mean_c - mean_overall).reshape(n_features, 1)
    S_B += n_c * (mean_diff @ mean_diff.T)

In [23]:
# Solve eigenvalue
# Add small regularization for stability
S_W += np.eye(n_features) * 1e-6
eigvals, eigvecs = np.linalg.eig(np.linalg.pinv(S_W).dot(S_B))

In [24]:
# Sort eigenvectors
sorted_indices = np.argsort(abs(eigvals))[::-1]
eigvecs = eigvecs[:, sorted_indices]

In [25]:
# Select top 39 components
W = eigvecs[:, :39].real

In [26]:
# Transform data
X_train_lda = X_train_scaled @ W
X_test_lda = X_test_scaled @ W

In [27]:
# Random forest after LDA
clf_lda = RandomForestClassifier(n_estimators=100, random_state=42)
clf_lda.fit(X_train_lda, y_train)
print("Accuracy with Manual LDA:",
      metrics.accuracy_score(y_test, clf_lda.predict(X_test_lda)))

Accuracy with Manual LDA: 0.1


In [28]:
# Logistic regression after LDA
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train_lda, y_train)
print("Accuracy Logistic Regression (Manual LDA):",
      metrics.accuracy_score(y_test, lr.predict(X_test_lda)))

Accuracy Logistic Regression (Manual LDA): 0.225


## Why Was Manual LDA Accuracy Low?

The Olivetti dataset contains 4096 features but only about 320 training samples across 40 classes.

For LDA:
rank(S_W) <= N - C = 320 - 40 = 280

Since 4096 >> 280, the within-class scatter matrix (S_W) becomes singular (non-invertible).

In the manual implementation, we computed:
S_W^-1 S_B

Because S_W is singular, using a pseudo-inverse introduces numerical instability and amplifies noise. This results in poor discriminative directions and low classification accuracy.

The sklearn LDA performed better because it uses SVD internally, which avoids direct matrix inversion and is numerically stable in high-dimensional settings.
